[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB-X3_mc_dropout_uncertainty.ipynb)

# NB-X3 · Challenge: uncertainty with Monte-Carlo dropout and abstention
**Day 5 · optional · about 40 minutes.** A model that can say *"I am not sure, ask a human"* is safer than one that always answers. We estimate uncertainty by keeping dropout **on** at test time and averaging many forward passes, then measure whether **abstaining on the most uncertain cases** improves accuracy on the rest.

In [ ]:
%pip install -q "medmnist==3.0.2"
import os, random, numpy as np, torch, torch.nn as nn, matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
def set_seed(s=0):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s); torch.backends.cudnn.deterministic = True
set_seed(0); device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); os.makedirs("data", exist_ok=True)
from medmnist import PneumoniaMNIST
D = {s: PneumoniaMNIST(split=s, download=True, root="data") for s in ("train", "val", "test")}
X = {s: torch.from_numpy(D[s].imgs.astype(np.float32) / 255).unsqueeze(1) for s in D}; Y = {s: D[s].labels.ravel().astype(np.int64) for s in D}

In [ ]:
def build(p=0.3):
    return nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Flatten(), nn.Dropout(p), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 1))
def train(epochs=8, seed=0):
    set_seed(seed); m = build().to(device); opt = torch.optim.Adam(m.parameters(), 1e-3); crit = nn.BCEWithLogitsLoss()
    dl = DataLoader(TensorDataset(X["train"], torch.from_numpy(Y["train"])), batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(seed))
    for _ in range(epochs):
        m.train()
        for xb, yb in dl: xb, yb = xb.to(device), yb.float().unsqueeze(1).to(device); opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
    return m
model = train()

## 1 · MC dropout
**TODO 1:** `mc_predict(model, x, T)` returns an array (T, N) of probabilities from T stochastic passes with dropout **active** and everything else in eval mode. Hint: `model.eval()` then switch only the `nn.Dropout` modules back with `.train()`. Remember to restore `eval()` afterwards.

In [ ]:
@torch.no_grad()
def mc_predict(model, x, T=30, bs=512):
    # TODO 1: eval mode, then .train() on the nn.Dropout modules only; T passes; stack into (T, N); restore eval()
    return np.zeros((T, len(x)))

In [ ]:
set_seed(0); P = mc_predict(model, X["val"], T=20)
assert P.shape == (20, len(X["val"]))
assert P.std(axis=0).mean() > 1e-4, "passes are identical: dropout was not active"
assert not any(m.training for m in model.modules()), "restore model.eval() at the end"
print("Checkpoint 1 passed | mean predictive std:", round(float(P.std(axis=0).mean()), 4))

## 2 · Uncertainty score and abstention
Mean probability `p̄` is the prediction; the **uncertainty** is the predictive entropy of `p̄` (high when `p̄` is near 0.5). **TODO 2:** `entropy(p)` for a binary probability, then `accuracy_vs_coverage(p̄, y, u)` for coverage 100%, 90%, …, 50% (keep the *most certain* cases).

In [ ]:
def entropy(p, eps=1e-8):
    p = np.clip(p, eps, 1 - eps)
    return np.zeros_like(p)      # TODO 2a: -(p log p + (1-p) log(1-p))
def accuracy_vs_coverage(pbar, y, u, coverages=(1.0, 0.9, 0.8, 0.7, 0.6, 0.5)):
    rows = []
    # TODO 2b: sort by uncertainty u ascending, keep the first round(c*N) cases, append (c, accuracy of (pbar>=0.5) on them)
    return rows

In [ ]:
yv = Y["val"]; pbar = P.mean(0); u = entropy(pbar)
assert abs(entropy(np.array([0.5]))[0] - np.log(2)) < 1e-6 and entropy(np.array([0.999]))[0] < 0.02
rows = accuracy_vs_coverage(pbar, yv, u); assert len(rows) == 6 and abs(rows[0][0] - 1.0) < 1e-9
for c, a in rows: print(f"coverage {c:.0%}: accuracy {a:.3f}")
assert rows[-1][1] >= rows[0][1] - 1e-9, "abstaining on the most uncertain cases should not lower accuracy"
plt.plot([r[0] for r in rows], [r[1] for r in rows], "o-"); plt.gca().invert_xaxis(); plt.xlabel("coverage (share of cases answered)"); plt.ylabel("accuracy on answered cases"); plt.show()
print("Checkpoint 2 passed")

## 3 · Plus: is the uncertainty useful on shifted data? 
Add noise (sd 0.15) to the validation images. **TODO 3:** compare the mean uncertainty on clean vs noisy images and the accuracy at 100% / 70% coverage on the noisy set. Does the uncertainty go up? Does abstention rescue the accuracy?

In [ ]:
# TODO 3: build the noisy set (clamp to 0..1), run mc_predict, compare mean entropy and accuracy_vs_coverage at 100% and 70%
g = torch.Generator().manual_seed(0)
print("write your code here")

**Reflection.** 1) MC dropout is cheap but approximate; name one thing it cannot capture (confident-but-wrong, out-of-distribution inputs). 2) Who handles the abstained cases, and what happens to workload if you abstain on 30%? 3) A model can be *calibrated* and still be *uncertain* on the wrong cases: how do ECE (NB06) and this notebook differ? 4) Would you report coverage and accuracy-on-answered together? Why never one alone?